 # Faster R-CNN on VOC — Staged Fine-Tuning Experiments



 **Pipeline:**

 1. **Stage 0** — Head warmup (4 epochs, frozen backbone). Shared launch point.

 2. **Stage 1** — Full fine-tuning with augmentation + noise + balanced loss.

 3. **Stage 1 (plain)** — Full fine-tuning ablation, no extras. Tests whether the tricks help.

 4. **Stage 2** — Incremental gradual unfreezing with augmentation + noise + balanced loss.

 5. **Test evaluation** — All three models on the held-out test split.



 Stages 1, 1-plain, and 2 all branch from the same Stage 0 checkpoint.

 ## CELL 0 — Setup (run once per session)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 0  |  SETUP
# ══════════════════════════════════════════════════════════════════════
from __future__ import annotations

import gc, json, random, subprocess, sys, time
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
import torchvision.transforms.functional as TF
from torchvision.ops import box_iou

for pkg in ["torchmetrics"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from torchmetrics.detection.mean_ap import MeanAveragePrecision

import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ── PATHS ─────────────────────────────────────────────────────────────
# Google Colab: uncomment the next two lines and set DATA_ROOT to your Drive path
# from google.colab import drive
# drive.mount('/content/drive')

DATA_ROOT = Path("/content/version_2/data")   # ← change if needed

CLEAN_IDS = DATA_ROOT / "clean_ids"
IMAGES    = DATA_ROOT / "images"
ANNS      = DATA_ROOT / "annotations"
CKPT_DIR  = Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

VOC_CLASSES = (
    "__background__",
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
)
CLASS_TO_IDX = {c: i for i, c in enumerate(VOC_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(VOC_CLASSES)  # 21

VOC_MEAN = [0.485, 0.456, 0.406]
VOC_STD  = [0.229, 0.224, 0.225]

# Shared paths for staged checkpoints
STAGE0_CKPT  = CKPT_DIR / "stage0_warmup.pth"
STAGE1_CKPT  = CKPT_DIR / "stage1_full_finetune_best.pth"
STAGE1P_CKPT = CKPT_DIR / "stage1_plain_best.pth"
STAGE2_CKPT  = CKPT_DIR / "stage2_incremental_best.pth"

# ── I/O helpers ───────────────────────────────────────────────────────
def read_ids(path: Path) -> List[str]:
    raw  = path.read_bytes().decode("utf-8-sig")
    seen, out = set(), []
    for ln in raw.splitlines():
        tok = ln.strip().split()[0] if ln.strip() else ""
        if tok and tok not in seen:
            seen.add(tok)
            out.append(tok)
    return out


def parse_xml(xml_path: Path):
    """Return (boxes, labels, difficult, truncated, occluded, img_w, img_h)."""
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    img_w = int(size.find("width").text)  if size is not None else 0
    img_h = int(size.find("height").text) if size is not None else 0
    boxes, labels = [], []
    difficult, truncated, occluded = [], [], []
    for obj in root.findall("object"):
        name_el = obj.find("name")
        if name_el is None or name_el.text not in CLASS_TO_IDX:
            continue
        bb = obj.find("bndbox")
        if bb is None:
            continue
        xmin = float(bb.find("xmin").text)
        ymin = float(bb.find("ymin").text)
        xmax = float(bb.find("xmax").text)
        ymax = float(bb.find("ymax").text)
        if xmax <= xmin or ymax <= ymin:
            continue
        boxes.append([xmin, ymin, xmax, ymax])
        labels.append(CLASS_TO_IDX[name_el.text])
        diff_el  = obj.find("difficult")
        trunc_el = obj.find("truncated")
        occ_el   = obj.find("occluded")
        difficult.append(int(diff_el.text)  if diff_el  is not None else 0)
        truncated.append(int(trunc_el.text) if trunc_el is not None else 0)
        occluded.append(int(occ_el.text)    if occ_el   is not None else 0)
    return boxes, labels, difficult, truncated, occluded, img_w, img_h


# ── Dataset ───────────────────────────────────────────────────────────
class VOCDataset(Dataset):
    def __init__(self, split: str, transform=None):
        self.ids       = read_ids(CLEAN_IDS / f"{split}.txt")
        self.split     = split
        self.transform = transform
        self.img_dir   = IMAGES / split
        self.ann_dir   = ANNS   / split

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx: int):
        img_id = self.ids[idx]
        img    = Image.open(self.img_dir / f"{img_id}.jpg").convert("RGB")
        boxes, labels, diff, trunc, occ, w, h = parse_xml(
            self.ann_dir / f"{img_id}.xml"
        )
        if boxes:
            boxes_t  = torch.tensor(boxes,  dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
            diff_t   = torch.tensor(diff,   dtype=torch.int64)
            trunc_t  = torch.tensor(trunc,  dtype=torch.int64)
            occ_t    = torch.tensor(occ,    dtype=torch.int64)
        else:
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,),   dtype=torch.int64)
            diff_t   = torch.zeros((0,),   dtype=torch.int64)
            trunc_t  = torch.zeros((0,),   dtype=torch.int64)
            occ_t    = torch.zeros((0,),   dtype=torch.int64)

        target = {
            "boxes":     boxes_t,
            "labels":    labels_t,
            "image_id":  torch.tensor([idx]),
            "difficult": diff_t,
            "truncated": trunc_t,
            "occluded":  occ_t,
            "orig_size": torch.tensor([h, w]),
        }
        if self.transform:
            img, target = self.transform(img, target)
        else:
            img = TF.to_tensor(img)
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


def _model_targets(targets):
    """Strip non-standard keys before passing to Faster R-CNN."""
    keep = {"boxes", "labels"}
    return [{k: v for k, v in t.items() if k in keep} for t in targets]


# ── Transforms ────────────────────────────────────────────────────────
class DetectionTransform:
    """
    mode = 'baseline' → ToTensor + Normalize only
    mode = 'full_aug' → + HFlip(0.5) + ColorJitter + Gaussian noise
    train=False disables all augmentation regardless of mode.
    """
    _jitter = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1)

    def __init__(self, mode: str = "baseline", train: bool = True):
        assert mode in ("baseline", "full_aug")
        self.mode  = mode
        self.train = train

    def __call__(self, img: Image.Image, target: dict):
        img_t = TF.to_tensor(img)

        if self.train and self.mode == "full_aug":
            # Horizontal flip
            if random.random() < 0.5:
                img_t = TF.hflip(img_t)
                if target["boxes"].numel() > 0:
                    w = img_t.shape[-1]
                    b = target["boxes"].clone()
                    b[:, [0, 2]] = w - b[:, [2, 0]]
                    target["boxes"] = b

            # ColorJitter (PIL round-trip)
            pil   = TF.to_pil_image(img_t)
            pil   = self._jitter(pil)
            img_t = TF.to_tensor(pil)

            # Gaussian noise
            noise = torch.randn_like(img_t) * 0.02
            img_t = torch.clamp(img_t + noise, 0.0, 1.0)

        img_t = TF.normalize(img_t, VOC_MEAN, VOC_STD)
        return img_t, target


# ── Class-balanced loss (monkey-patch on torchvision roi_heads) ───────
import torchvision.models.detection.roi_heads as _rh_module

_ORIG_FASTRCNN_LOSS = _rh_module.fastrcnn_loss


def _make_balanced_loss_fn(class_weights: torch.Tensor):
    w = class_weights.clone()

    def _balanced_loss(class_logits, box_regression, labels, regression_targets):
        labels_cat  = torch.cat(labels, dim=0)
        reg_tgt_cat = torch.cat(regression_targets, dim=0)
        device      = class_logits.device

        cls_loss = F.cross_entropy(class_logits, labels_cat, weight=w.to(device))

        pos_idx    = torch.where(labels_cat > 0)[0]
        labels_pos = labels_cat[pos_idx]
        N, _       = class_logits.shape
        box_regression = box_regression.reshape(N, box_regression.size(-1) // 4, 4)

        box_loss = F.smooth_l1_loss(
            box_regression[pos_idx, labels_pos],
            reg_tgt_cat[pos_idx],
            beta=1.0 / 9,
            reduction="sum",
        ) / max(1, labels_cat.numel())

        return cls_loss, box_loss

    return _balanced_loss


def patch_balanced_loss(class_weights: torch.Tensor):
    _rh_module.fastrcnn_loss = _make_balanced_loss_fn(class_weights)


def restore_original_loss():
    _rh_module.fastrcnn_loss = _ORIG_FASTRCNN_LOSS


def compute_class_weights(split: str = "train") -> torch.Tensor:
    ids    = read_ids(CLEAN_IDS / f"{split}.txt")
    counts = Counter()
    for img_id in ids:
        _, labels, *_ = parse_xml(ANNS / split / f"{img_id}.xml")
        counts.update(labels)
    total   = sum(counts.values()) or 1
    weights = torch.ones(NUM_CLASSES, dtype=torch.float32)
    for cls_idx, cnt in counts.items():
        if cnt > 0:
            weights[cls_idx] = total / (NUM_CLASSES * cnt)
    weights[0] = weights[1:].mean()
    return weights


# ── Model + freeze helpers ────────────────────────────────────────────
def get_model() -> torchvision.models.detection.FasterRCNN:
    weights  = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model    = fasterrcnn_resnet50_fpn_v2(weights=weights)
    in_feats = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feats, NUM_CLASSES)
    return model


def freeze_all_backbone(model):
    """Freeze the entire backbone (body + FPN). Head + RPN remain trainable."""
    for p in model.backbone.parameters():
        p.requires_grad_(True)  # reset first
    for p in model.backbone.parameters():
        p.requires_grad_(False)


def set_grad(module, requires: bool):
    for p in module.parameters():
        p.requires_grad_(requires)


def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad_(True)


def trainable_param_count(model) -> Tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total



# ── Training / eval ───────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, device, scaler=None) -> float:
    model.train()
    total = 0.0
    num_batches = len(loader)

    for i, (images, targets) in enumerate(loader):
        imgs = [img.to(device) for img in images]
        tgts = [{k: v.to(device) for k, v in t.items()}
                for t in _model_targets(targets)]

        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            loss_dict = model(imgs, tgts)
            loss      = sum(loss_dict.values())

        optimizer.zero_grad()
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 5.0
            )
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 5.0
            )
            optimizer.step()

        total += loss.item()


        if (i + 1) % 50 == 0 or (i + 1) == num_batches:
            print(f"\r      Batch {i + 1:>4}/{num_batches} | Running Loss: {total / (i + 1):.4f}", end="", flush=True)

    print() # Moves to a new line when the epoch is fully done
    return total / max(1, len(loader))

@torch.no_grad()
def evaluate_map(model, loader, device) -> dict:
    model.eval()
    metric = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)
    for images, targets in loader:
        imgs  = [img.to(device) for img in images]
        preds = model(imgs)
        metric.update(
            [{"boxes": p["boxes"].cpu(), "scores": p["scores"].cpu(),
              "labels": p["labels"].cpu()} for p in preds],
            [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()}
             for t in targets],
        )
    return metric.compute()


@torch.no_grad()
def evaluate_person_pr(model, loader, device, iou_threshold: float = 0.5):
    person_idx = CLASS_TO_IDX["person"]
    model.eval()
    tp = fp = fn = 0
    for images, targets in loader:
        imgs  = [img.to(device) for img in images]
        preds = model(imgs)
        for pred, tgt in zip(preds, targets):
            p_mask  = pred["labels"] == person_idx
            g_mask  = tgt["labels"] == person_idx
            p_boxes = pred["boxes"][p_mask].cpu()
            p_scores= pred["scores"][p_mask].cpu()
            g_boxes = tgt["boxes"][g_mask].cpu()

            if p_boxes.numel() > 0:
                order   = p_scores.argsort(descending=True)
                p_boxes = p_boxes[order]

            matched: set = set()
            for pb in p_boxes:
                if g_boxes.numel() > 0:
                    ious = box_iou(pb.unsqueeze(0), g_boxes)[0]
                    best_iou, best_j = ious.max(0)
                    if best_iou.item() >= iou_threshold and best_j.item() not in matched:
                        tp += 1
                        matched.add(best_j.item())
                    else:
                        fp += 1
                else:
                    fp += 1
            fn += g_boxes.shape[0] - len(matched)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return precision, recall


# ── Early stopping helper ─────────────────────────────────────────────
class EarlyStopper:
    """Stops when val metric hasn't improved for `patience` epochs."""
    def __init__(self, patience: int = 3, min_delta: float = 1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.best      = -float("inf")
        self.bad       = 0

    def step(self, value: float) -> Tuple[bool, bool]:
        """Returns (improved, should_stop)."""
        improved = value > self.best + self.min_delta
        if improved:
            self.best = value
            self.bad  = 0
        else:
            self.bad += 1
        return improved, self.bad >= self.patience


# ── Loader factory ────────────────────────────────────────────────────
def make_loaders(transform_mode: str, batch_size: int = 2):
    train_ds = VOCDataset("train", DetectionTransform(transform_mode, train=True))
    val_ds   = VOCDataset("val",   DetectionTransform("baseline",     train=False))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=2, collate_fn=collate_fn,
                          pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)
    return train_ds, val_ds, train_dl, val_dl


print("\nSetup complete.")
print(f"NUM_CLASSES = {NUM_CLASSES}")
print(f"Splits available: {[p.name for p in CLEAN_IDS.glob('*.txt')]}")



Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB

Setup complete.
NUM_CLASSES = 21
Splits available: ['split_seed.txt', 'dev_test.txt', 'val.txt', 'test.txt', 'train.txt']


 ## CELL 1 — Dataset Statistics

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1  |  DATASET STATISTICS
# ══════════════════════════════════════════════════════════════════════

def compute_dataset_statistics(split: str = "train") -> None:
    ids     = read_ids(CLEAN_IDS / f"{split}.txt")
    ann_dir = ANNS / split
    print(f"\n{'═'*62}")
    print(f"  Dataset statistics — split: {split}  (n={len(ids)})")
    print(f"{'═'*62}")

    img_cls_count = Counter()
    inst_count    = Counter()
    total_diff = total_trunc = total_occ = total_objs = 0
    aspects, areas_norm, objs_per_img = [], [], []

    for img_id in ids:
        boxes, labels, diff, trunc, occ, w, h = parse_xml(
            ann_dir / f"{img_id}.xml"
        )
        img_area = max(1, w * h)
        seen_cls: set = set()
        objs_per_img.append(len(labels))
        for i, lbl in enumerate(labels):
            seen_cls.add(lbl)
            inst_count[lbl] += 1
            x1, y1, x2, y2 = boxes[i]
            bw  = max(1, x2 - x1)
            bh  = max(1, y2 - y1)
            aspects.append(bw / bh)
            areas_norm.append((bw * bh) / img_area)
            total_diff  += diff[i]
            total_trunc += trunc[i]
            total_occ   += occ[i]
            total_objs  += 1
        for c in seen_cls:
            img_cls_count[c] += 1

    print("\n[1] Per-class image & instance counts")
    print(f"  {'Class':<16} {'Imgs':>6} {'Instances':>10} {'Img%':>7} {'Inst%':>7}")
    print(f"  {'-'*51}")
    total_imgs = len(ids)
    sorted_cls = sorted(range(1, 21), key=lambda c: inst_count[c], reverse=True)
    for c in sorted_cls:
        cls_name = VOC_CLASSES[c]
        n_img    = img_cls_count[c]
        n_inst   = inst_count[c]
        print(f"  {cls_name:<16} {n_img:>6} {n_inst:>10} "
              f"{n_img/total_imgs*100:>6.1f}% {n_inst/max(1,total_objs)*100:>6.1f}%")

    counts_fg = [inst_count[c] for c in range(1, 21)]
    imbalance = max(counts_fg) / max(1, min(counts_fg))
    print(f"\n  → Max/min instance ratio (class imbalance): {imbalance:.1f}×")

    aspects_arr = np.array(aspects)
    areas_arr   = np.array(areas_norm)
    print("\n[2] Bounding box geometry")
    print(f"  Aspect ratio (w/h)  — mean={aspects_arr.mean():.2f}  "
          f"std={aspects_arr.std():.2f}  "
          f"p5={np.percentile(aspects_arr,5):.2f}  "
          f"p95={np.percentile(aspects_arr,95):.2f}")
    print(f"  Normalised area     — mean={areas_arr.mean():.3f}  "
          f"std={areas_arr.std():.3f}  "
          f"p5={np.percentile(areas_arr,5):.4f}  "
          f"p95={np.percentile(areas_arr,95):.3f}")
    tiny  = (areas_arr < 0.01).mean() * 100
    large = (areas_arr > 0.25).mean() * 100
    print(f"  Tiny objects  (<1% img area):  {tiny:.1f}%")
    print(f"  Large objects (>25% img area): {large:.1f}%")

    opi = np.array(objs_per_img)
    print("\n[3] Objects per image")
    print(f"  mean={opi.mean():.2f}  std={opi.std():.2f}  "
          f"max={opi.max()}  images_with_0={int((opi==0).sum())}")

    print("\n[4] Non-ideal characteristics (flags)")
    print(f"  Difficult  : {total_diff:>6} / {total_objs}  "
          f"({total_diff/max(1,total_objs)*100:.1f}%)")
    print(f"  Truncated  : {total_trunc:>6} / {total_objs}  "
          f"({total_trunc/max(1,total_objs)*100:.1f}%)")
    print(f"  Occluded   : {total_occ:>6} / {total_objs}  "
          f"({total_occ/max(1,total_objs)*100:.1f}%)")
    print(f"{'═'*62}")


compute_dataset_statistics("train")




══════════════════════════════════════════════════════════════
  Dataset statistics — split: train  (n=5715)
══════════════════════════════════════════════════════════════

[1] Per-class image & instance counts
  Class              Imgs  Instances    Img%   Inst%
  ---------------------------------------------------
  person             2142       5019   37.5%   31.8%
  chair               656       1457   11.5%    9.2%
  car                 621       1191   10.9%    7.6%
  dog                 636        768   11.1%    4.9%
  bottle              399        749    7.0%    4.7%
  cat                 540        609    9.4%    3.9%
  bird                399        592    7.0%    3.8%
  pottedplant         289        557    5.1%    3.5%
  sheep               171        509    3.0%    3.2%
  boat                264        508    4.6%    3.2%
  aeroplane           326        468    5.7%    3.0%
  tvmonitor           299        412    5.2%    2.6%
  bicycle             281        410    4.9% 

 ## CELL 2 — Stage 0: Head Warmup (4 epochs, no early stopping)



 Trains only the RPN + ROI heads with backbone + FPN frozen.

 Uses **baseline** transform (no aug) — goal is just to settle the random box predictor

 before any downstream stage touches the backbone.



 Saves the **final epoch** weights (not best) to `checkpoints/stage0_warmup.pth`

 so all downstream stages launch from identical weights.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2  |  STAGE 0 — HEAD WARMUP
# ══════════════════════════════════════════════════════════════════════

STAGE0_EPOCHS     = 4
STAGE0_BATCH_SIZE = 4
STAGE0_LR         = 1e-4

print(f"\n{'═'*62}")
print(f"  STAGE 0 — Head Warmup")
print(f"  epochs={STAGE0_EPOCHS}  lr={STAGE0_LR}  transform=baseline")
print(f"{'═'*62}")

train_ds, val_ds, train_dl, val_dl = make_loaders("baseline", STAGE0_BATCH_SIZE)

model = get_model().to(DEVICE)
freeze_all_backbone(model)

trn, tot = trainable_param_count(model)
print(f"  Trainable params: {trn/1e6:.2f}M / {tot/1e6:.2f}M "
      f"({trn/tot*100:.1f}%)")

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=STAGE0_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=STAGE0_EPOCHS, eta_min=STAGE0_LR * 0.1
)
scaler = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

best_val = 0.0
for epoch in range(1, STAGE0_EPOCHS + 1):
    t0    = time.time()
    loss  = train_one_epoch(model, train_dl, optimizer, DEVICE, scaler)
    res   = evaluate_map(model, val_dl, DEVICE)
    map50 = float(res["map_50"])
    scheduler.step()
    best_val = max(best_val, map50)
    print(f"  epoch {epoch}/{STAGE0_EPOCHS}  loss={loss:.4f}  "
          f"val_mAP@0.5={map50:.4f}  ({time.time()-t0:.0f}s)")

# Save FINAL epoch (not best) — this is the shared launch checkpoint
torch.save(
    {"epoch": STAGE0_EPOCHS,
     "model_state": model.state_dict(),
     "val_map50_final": map50,
     "val_map50_best":  best_val,
     "config": {"stage": "stage0_warmup", "transform": "baseline"}},
    STAGE0_CKPT,
)
print(f"\n  Saved final-epoch warmup checkpoint → {STAGE0_CKPT}")
print(f"  (final val mAP@0.5={map50:.4f}  best-during-warmup={best_val:.4f})")

del model, train_dl, val_dl, train_ds, val_ds, optimizer, scheduler, scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()




══════════════════════════════════════════════════════════════
  STAGE 0 — Head Warmup
  epochs=4  lr=0.0001  transform=baseline
══════════════════════════════════════════════════════════════
  Trainable params: 16.50M / 43.35M (38.1%)


/tmp/ipykernel_4728/1429214447.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
/tmp/ipykernel_4728/3030994879.py:317: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


      Batch 1428/1428 | Running Loss: 0.3083
  epoch 1/4  loss=0.3083  val_mAP@0.5=0.8190  (1167s)
      Batch 1428/1428 | Running Loss: 0.2345
  epoch 2/4  loss=0.2345  val_mAP@0.5=0.8270  (1170s)
      Batch 1428/1428 | Running Loss: 0.2060
  epoch 3/4  loss=0.2060  val_mAP@0.5=0.8276  (1169s)
      Batch 1428/1428 | Running Loss: 0.1814
  epoch 4/4  loss=0.1814  val_mAP@0.5=0.8466  (1165s)

  Saved final-epoch warmup checkpoint → checkpoints/stage0_warmup.pth
  (final val mAP@0.5=0.8466  best-during-warmup=0.8466)


 ## CELL 3 — Stage 1: Full Fine-Tuning (with augmentation + noise + balanced loss)



 Loads `stage0_warmup.pth`, unfreezes everything, trains for up to 10 epochs

 with early stopping (patience=3). Uses the full augmentation stack

 (HFlip + ColorJitter + Gaussian noise) and class-balanced cross-entropy.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3  |  STAGE 1 — FULL FINE-TUNING (full extras)
# ══════════════════════════════════════════════════════════════════════

STAGE1_MAX_EPOCHS = 10
STAGE1_BATCH_SIZE = 2
STAGE1_LR         = 5e-5
STAGE1_PATIENCE   = 3

assert STAGE0_CKPT.is_file(), f"Run Cell 2 first — missing {STAGE0_CKPT}"

print(f"\n{'═'*62}")
print(f"  STAGE 1 — Full fine-tuning (aug + noise + balanced loss)")
print(f"  max_epochs={STAGE1_MAX_EPOCHS}  lr={STAGE1_LR}  "
      f"patience={STAGE1_PATIENCE}")
print(f"{'═'*62}")

cw = compute_class_weights("train")
print(f"  Class weights — fg range [{cw[1:].min():.3f}, {cw[1:].max():.3f}]  "
      f"bg={cw[0]:.3f}")
patch_balanced_loss(cw)

train_ds, val_ds, train_dl, val_dl = make_loaders("full_aug", STAGE1_BATCH_SIZE)

model = get_model().to(DEVICE)
ckpt0 = torch.load(STAGE0_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt0["model_state"])
print(f"  Loaded warmup checkpoint (val_mAP@0.5={ckpt0['val_map50_final']:.4f})")
unfreeze_all(model)

trn, tot = trainable_param_count(model)
print(f"  Trainable params: {trn/1e6:.2f}M / {tot/1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=STAGE1_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=STAGE1_MAX_EPOCHS, eta_min=STAGE1_LR * 0.1
)
scaler   = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
stopper  = EarlyStopper(patience=STAGE1_PATIENCE)
best_val = 0.0

for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
    t0    = time.time()
    loss  = train_one_epoch(model, train_dl, optimizer, DEVICE, scaler)
    res   = evaluate_map(model, val_dl, DEVICE)
    map50 = float(res["map_50"])
    scheduler.step()

    improved, stop = stopper.step(map50)
    flag = "  ✓ best" if improved else ""
    print(f"  epoch {epoch:>2}/{STAGE1_MAX_EPOCHS}  loss={loss:.4f}  "
          f"val_mAP@0.5={map50:.4f}  ({time.time()-t0:.0f}s){flag}")

    if improved:
        best_val = map50
        torch.save(
            {"epoch": epoch,
             "model_state": model.state_dict(),
             "map50": map50,
             "config": {"stage": "stage1_full_finetune",
                        "transform": "full_aug",
                        "balanced_loss": True}},
            STAGE1_CKPT,
        )

    if stop:
        print(f"  ⏹ Early stopping (no improvement for {STAGE1_PATIENCE} epochs)")
        break

restore_original_loss()
print(f"\n  Stage 1 best val mAP@0.5 = {best_val:.4f}")
print(f"  Checkpoint → {STAGE1_CKPT}")

del model, train_dl, val_dl, train_ds, val_ds, optimizer, scheduler, scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()




══════════════════════════════════════════════════════════════
  STAGE 1 — Full fine-tuning (aug + noise + balanced loss)
  max_epochs=10  lr=5e-05  patience=3
══════════════════════════════════════════════════════════════
  Class weights — fg range [0.150, 2.369]  bg=1.501
  Loaded warmup checkpoint (val_mAP@0.5=0.8466)
  Trainable params: 43.35M / 43.35M


/tmp/ipykernel_52838/3646021826.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler   = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
/tmp/ipykernel_52838/3030994879.py:317: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


      Batch 2857/2857 | Running Loss: 0.2674
  epoch  1/10  loss=0.2674  val_mAP@0.5=0.7258  (1663s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.2424
  epoch  2/10  loss=0.2424  val_mAP@0.5=0.7281  (1654s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.2174
  epoch  3/10  loss=0.2174  val_mAP@0.5=0.7391  (1657s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1945
  epoch  4/10  loss=0.1945  val_mAP@0.5=0.7374  (1655s)
      Batch 2857/2857 | Running Loss: 0.1719
  epoch  5/10  loss=0.1719  val_mAP@0.5=0.7580  (1656s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1489
  epoch  6/10  loss=0.1489  val_mAP@0.5=0.7608  (1655s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1284
  epoch  7/10  loss=0.1284  val_mAP@0.5=0.7680  (1649s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1137
  epoch  8/10  loss=0.1137  val_mAP@0.5=0.7731  (1652s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1004
  epoch  9/10  loss=0.1004  val_mAP@0.5=0.7705  (1648s)
      Batch 2857/2857 | Running Los

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3.5 | STAGE 1 EXTENSION — Continue Training
# ══════════════════════════════════════════════════════════════════════

EXTRA_EPOCHS   = 10
STAGE1_LR_CONT = 1e-5  # Lower LR since the model is already well-trained
CONT_PATIENCE  = 6

print(f"\n{'═'*62}")
print(f"  STAGE 1 EXTENSION — Continuing from best checkpoint")
print(f"  extra_epochs={EXTRA_EPOCHS}  lr={STAGE1_LR_CONT}")
print(f"{'═'*62}")

# 1. Rebuild loaders and patch the loss function again
cw = compute_class_weights("train")
patch_balanced_loss(cw)
train_ds, val_ds, train_dl, val_dl = make_loaders("full_aug", batch_size=2)

# 2. Load the Best Model from Stage 1
model = get_model().to(DEVICE)
ckpt  = torch.load(STAGE1_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
unfreeze_all(model)

best_val    = float(ckpt["map50"])
start_epoch = ckpt["epoch"] + 1
print(f"  Resuming from Epoch {start_epoch-1} (Previous Best val_mAP@0.5 = {best_val:.4f})")

# 3. Setup Optimizer, Scaler, and Stopper
optimizer = torch.optim.AdamW(model.parameters(), lr=STAGE1_LR_CONT, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EXTRA_EPOCHS, eta_min=1e-6
)
scaler  = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

# Crucial: Tell the EarlyStopper what the previous best score was!
stopper      = EarlyStopper(patience=CONT_PATIENCE)
stopper.best = best_val

# 4. Continue the Loop
for epoch in range(start_epoch, start_epoch + EXTRA_EPOCHS):
    t0    = time.time()
    loss  = train_one_epoch(model, train_dl, optimizer, DEVICE, scaler)
    res   = evaluate_map(model, val_dl, DEVICE)
    map50 = float(res["map_50"])
    scheduler.step()

    improved, stop = stopper.step(map50)
    flag = "  ✓ NEW best!" if improved else ""
    print(f"  epoch {epoch:>2}/{start_epoch + EXTRA_EPOCHS - 1}  loss={loss:.4f}  "
          f"val_mAP@0.5={map50:.4f}  ({time.time()-t0:.0f}s){flag}")

    if improved:
        best_val = map50
        torch.save(
            {"epoch": epoch,
             "model_state": model.state_dict(),
             "map50": map50,
             "config": {"stage": "stage1_full_finetune_extended",
                        "transform": "full_aug",
                        "balanced_loss": True}},
            STAGE1_CKPT, # Overwrites the previous best checkpoint safely
        )

    if stop:
        print(f"  ⏹ Early stopping (no improvement for {CONT_PATIENCE} epochs)")
        break

restore_original_loss()
print(f"\n  Stage 1 Extended best val mAP@0.5 = {best_val:.4f}")
print(f"  Checkpoint → {STAGE1_CKPT}")

# Cleanup
del model, train_dl, val_dl, train_ds, val_ds, optimizer, scheduler, scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


══════════════════════════════════════════════════════════════
  STAGE 1 EXTENSION — Continuing from best checkpoint
  extra_epochs=10  lr=1e-05
══════════════════════════════════════════════════════════════
  Resuming from Epoch 10 (Previous Best val_mAP@0.5 = 0.7755)


/tmp/ipykernel_52838/1548617841.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
/tmp/ipykernel_52838/2263844319.py:317: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


      Batch 2857/2857 | Running Loss: 0.0904
  epoch 11/20  loss=0.0904  val_mAP@0.5=0.7689  (1652s)
      Batch 2857/2857 | Running Loss: 0.0857
  epoch 12/20  loss=0.0857  val_mAP@0.5=0.7652  (1652s)
      Batch 2857/2857 | Running Loss: 0.0821
  epoch 13/20  loss=0.0821  val_mAP@0.5=0.7705  (1649s)
      Batch 2857/2857 | Running Loss: 0.0782
  epoch 14/20  loss=0.0782  val_mAP@0.5=0.7673  (1652s)
      Batch 2857/2857 | Running Loss: 0.0733
  epoch 15/20  loss=0.0733  val_mAP@0.5=0.7574  (1645s)
      Batch 2857/2857 | Running Loss: 0.0697
  epoch 16/20  loss=0.0697  val_mAP@0.5=0.7608  (1651s)
  ⏹ Early stopping (no improvement for 6 epochs)

  Stage 1 Extended best val mAP@0.5 = 0.7755
  Checkpoint → checkpoints/stage1_full_finetune_best.pth


 ## CELL 4 — Stage 1 Plain: Full Fine-Tuning Ablation (no extras)



 Same as Cell 3 but with **no augmentation, no noise, no class balancing**.

 Tests whether the full extras stack actually helps over plain fine-tuning.

 Branches from the same Stage 0 checkpoint.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4  |  STAGE 1 PLAIN — Full fine-tuning ablation
# ══════════════════════════════════════════════════════════════════════

STAGE1_MAX_EPOCHS = 10
STAGE1_BATCH_SIZE = 2
STAGE1_LR         = 5e-5
STAGE1_PATIENCE   = 3

assert STAGE0_CKPT.is_file(), f"Run Cell 2 first — missing {STAGE0_CKPT}"

print(f"\n{'═'*62}")
print(f"  STAGE 1 PLAIN — Full fine-tuning (no extras)")
print(f"  max_epochs={STAGE1_MAX_EPOCHS}  lr={STAGE1_LR}  "
      f"patience={STAGE1_PATIENCE}")
print(f"{'═'*62}")

train_ds, val_ds, train_dl, val_dl = make_loaders("baseline", STAGE1_BATCH_SIZE)

model = get_model().to(DEVICE)
ckpt0 = torch.load(STAGE0_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt0["model_state"])
print(f"  Loaded warmup checkpoint (val_mAP@0.5={ckpt0['val_map50_final']:.4f})")
unfreeze_all(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=STAGE1_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=STAGE1_MAX_EPOCHS, eta_min=STAGE1_LR * 0.1
)
scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
stopper   = EarlyStopper(patience=STAGE1_PATIENCE)
best_val  = 0.0

for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
    t0    = time.time()
    loss  = train_one_epoch(model, train_dl, optimizer, DEVICE, scaler)
    res   = evaluate_map(model, val_dl, DEVICE)
    map50 = float(res["map_50"])
    scheduler.step()

    improved, stop = stopper.step(map50)
    flag = "  ✓ best" if improved else ""
    print(f"  epoch {epoch:>2}/{STAGE1_MAX_EPOCHS}  loss={loss:.4f}  "
          f"val_mAP@0.5={map50:.4f}  ({time.time()-t0:.0f}s){flag}")

    if improved:
        best_val = map50
        torch.save(
            {"epoch": epoch,
             "model_state": model.state_dict(),
             "map50": map50,
             "config": {"stage": "stage1_plain",
                        "transform": "baseline",
                        "balanced_loss": False}},
            STAGE1P_CKPT,
        )

    if stop:
        print(f"  ⏹ Early stopping (no improvement for {STAGE1_PATIENCE} epochs)")
        break

print(f"\n  Stage 1 plain best val mAP@0.5 = {best_val:.4f}")
print(f"  Checkpoint → {STAGE1P_CKPT}")

del model, train_dl, val_dl, train_ds, val_ds, optimizer, scheduler, scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()




══════════════════════════════════════════════════════════════
  STAGE 1 PLAIN — Full fine-tuning (no extras)
  max_epochs=10  lr=5e-05  patience=3
══════════════════════════════════════════════════════════════
  Loaded warmup checkpoint (val_mAP@0.5=0.8466)


/tmp/ipykernel_181/2219836139.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
/tmp/ipykernel_181/3548752969.py:317: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


      Batch 2857/2857 | Running Loss: 0.2763
  epoch  1/10  loss=0.2763  val_mAP@0.5=0.7232  (1661s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.2287
  epoch  2/10  loss=0.2287  val_mAP@0.5=0.7279  (1632s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1887
  epoch  3/10  loss=0.1887  val_mAP@0.5=0.7374  (1632s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1554
  epoch  4/10  loss=0.1554  val_mAP@0.5=0.7476  (1630s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1278
  epoch  5/10  loss=0.1278  val_mAP@0.5=0.7474  (1633s)
      Batch 2857/2857 | Running Loss: 0.1044
  epoch  6/10  loss=0.1044  val_mAP@0.5=0.7551  (1632s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.0859
  epoch  7/10  loss=0.0859  val_mAP@0.5=0.7630  (1628s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.0708
  epoch  8/10  loss=0.0708  val_mAP@0.5=0.7611  (1631s)
      Batch 2857/2857 | Running Loss: 0.0595
  epoch  9/10  loss=0.0595  val_mAP@0.5=0.7619  (1629s)
      Batch 2857/2857 | Running Loss: 0.051

 ## CELL 5 — Stage 2: Incremental Gradual Unfreezing



 Loads the **same** `stage0_warmup.pth` and gradually thaws the backbone over 10 epochs:



 | Epoch | Frozen → Unfrozen step              | LR     |

 |-------|--------------------------------------|--------|

 | 1–2   | (head + RPN trainable, all backbone frozen) | 5e-5 |

 | 3–4   | + FPN unfrozen                       | 5e-5  |

 | 5–6   | + backbone layer4                    | 2.5e-5|

 | 7–8   | + backbone layer3                    | 2.5e-5|

 | 9–10  | + backbone layer2 + layer1 (full)    | 1e-5  |



 Optimizer is rebuilt each time a new group thaws (so AdamW sees the new params).

 Augmentation + noise + balanced loss active throughout. Early stopping (patience=3).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5  |  STAGE 2 — INCREMENTAL GRADUAL UNFREEZING
# ══════════════════════════════════════════════════════════════════════

STAGE2_MAX_EPOCHS = 10
STAGE2_BATCH_SIZE = 2
STAGE2_PATIENCE   = 3

# (start_epoch, lr, description, unfreeze_action)
def _stage2_schedule(model):
    """Return list of (epoch_start, lr, description, action_callable)."""
    bb_body = model.backbone.body  # ResNet50 body: conv1, bn1, layer1, layer2, layer3, layer4
    fpn     = model.backbone.fpn

    def keep_frozen():
        # Backbone fully frozen — only head + RPN train
        set_grad(model.backbone, False)

    def thaw_fpn():
        set_grad(fpn, True)

    def thaw_layer4():
        set_grad(bb_body.layer4, True)

    def thaw_layer3():
        set_grad(bb_body.layer3, True)

    def thaw_layer1_2():
        set_grad(bb_body.layer2, True)
        set_grad(bb_body.layer1, True)
        # conv1/bn1 typically left frozen; uncomment to thaw:
        # set_grad(bb_body.conv1, True); set_grad(bb_body.bn1, True)

    return [
        (1, 5.0e-5, "head + RPN only (backbone frozen)", keep_frozen),
        (3, 5.0e-5, "+ FPN unfrozen",                    thaw_fpn),
        (5, 2.5e-5, "+ layer4 unfrozen",                 thaw_layer4),
        (7, 2.5e-5, "+ layer3 unfrozen",                 thaw_layer3),
        (9, 1.0e-5, "+ layer2 + layer1 unfrozen",        thaw_layer1_2),
    ]


assert STAGE0_CKPT.is_file(), f"Run Cell 2 first — missing {STAGE0_CKPT}"

print(f"\n{'═'*62}")
print(f"  STAGE 2 — Incremental gradual unfreezing")
print(f"  max_epochs={STAGE2_MAX_EPOCHS}  patience={STAGE2_PATIENCE}")
print(f"{'═'*62}")

cw = compute_class_weights("train")
print(f"  Class weights — fg range [{cw[1:].min():.3f}, {cw[1:].max():.3f}]")
patch_balanced_loss(cw)

train_ds, val_ds, train_dl, val_dl = make_loaders("full_aug", STAGE2_BATCH_SIZE)

model = get_model().to(DEVICE)
ckpt0 = torch.load(STAGE0_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt0["model_state"])
print(f"  Loaded warmup checkpoint (val_mAP@0.5={ckpt0['val_map50_final']:.4f})")

schedule = _stage2_schedule(model)

# Apply initial state (epoch 1: backbone fully frozen)
schedule[0][3]()
current_lr = schedule[0][1]
print(f"  [epoch 1] {schedule[0][2]}  lr={current_lr}")

trn, tot = trainable_param_count(model)
print(f"  Trainable params: {trn/1e6:.2f}M / {tot/1e6:.2f}M")

def _build_optim(lr):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)

optimizer = _build_optim(current_lr)
scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
stopper   = EarlyStopper(patience=STAGE2_PATIENCE)
best_val  = 0.0

for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
    # Apply any schedule transitions for this epoch (skip index 0, applied above)
    for (start_ep, lr, desc, action) in schedule[1:]:
        if epoch == start_ep:
            action()
            current_lr = lr
            optimizer  = _build_optim(current_lr)
            if DEVICE.type == "cuda":
                scaler = torch.cuda.amp.GradScaler()
            trn, tot = trainable_param_count(model)
            print(f"  [epoch {epoch}] {desc}  lr={current_lr}  "
                  f"trainable={trn/1e6:.2f}M ({trn/tot*100:.1f}%)")

    t0    = time.time()
    loss  = train_one_epoch(model, train_dl, optimizer, DEVICE, scaler)
    res   = evaluate_map(model, val_dl, DEVICE)
    map50 = float(res["map_50"])

    improved, stop = stopper.step(map50)
    flag = "  ✓ best" if improved else ""
    print(f"  epoch {epoch:>2}/{STAGE2_MAX_EPOCHS}  loss={loss:.4f}  "
          f"val_mAP@0.5={map50:.4f}  ({time.time()-t0:.0f}s){flag}")

    if improved:
        best_val = map50
        torch.save(
            {"epoch": epoch,
             "model_state": model.state_dict(),
             "map50": map50,
             "config": {"stage": "stage2_incremental",
                        "transform": "full_aug",
                        "balanced_loss": True,
                        "current_lr": current_lr}},
            STAGE2_CKPT,
        )

    if stop:
        print(f"  ⏹ Early stopping (no improvement for {STAGE2_PATIENCE} epochs)")
        break

restore_original_loss()
print(f"\n  Stage 2 best val mAP@0.5 = {best_val:.4f}")
print(f"  Checkpoint → {STAGE2_CKPT}")

del model, train_dl, val_dl, train_ds, val_ds, optimizer, scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()




══════════════════════════════════════════════════════════════
  STAGE 2 — Incremental gradual unfreezing
  max_epochs=10  patience=3
══════════════════════════════════════════════════════════════
  Class weights — fg range [0.150, 2.369]
  Loaded warmup checkpoint (val_mAP@0.5=0.8466)
  [epoch 1] head + RPN only (backbone frozen)  lr=5e-05
  Trainable params: 16.50M / 43.35M


/tmp/ipykernel_660/4222350433.py:76: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
/tmp/ipykernel_660/2263844319.py:317: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


      Batch 2857/2857 | Running Loss: 0.2341
  epoch  1/10  loss=0.2341  val_mAP@0.5=0.7945  (1158s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.2223
  epoch  2/10  loss=0.2223  val_mAP@0.5=0.7627  (1142s)
  [epoch 3] + FPN unfrozen  lr=5e-05  trainable=19.85M (45.8%)


/tmp/ipykernel_660/4222350433.py:88: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


      Batch 2857/2857 | Running Loss: 0.2264
  epoch  3/10  loss=0.2264  val_mAP@0.5=0.8001  (1314s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.2157
  epoch  4/10  loss=0.2157  val_mAP@0.5=0.7935  (1317s)
  [epoch 5] + layer4 unfrozen  lr=2.5e-05  trainable=34.81M (80.3%)
      Batch 2857/2857 | Running Loss: 0.2026
  epoch  5/10  loss=0.2026  val_mAP@0.5=0.7976  (1361s)
      Batch 2857/2857 | Running Loss: 0.1826
  epoch  6/10  loss=0.1826  val_mAP@0.5=0.8020  (1359s)  ✓ best
  [epoch 7] + layer3 unfrozen  lr=2.5e-05  trainable=41.91M (96.7%)
      Batch 2857/2857 | Running Loss: 0.1751
  epoch  7/10  loss=0.1751  val_mAP@0.5=0.7825  (1434s)
      Batch 2857/2857 | Running Loss: 0.1628
  epoch  8/10  loss=0.1628  val_mAP@0.5=0.7882  (1436s)
  [epoch 9] + layer2 + layer1 unfrozen  lr=1e-05  trainable=43.34M (100.0%)
      Batch 2857/2857 | Running Loss: 0.1369
  epoch  9/10  loss=0.1369  val_mAP@0.5=0.8087  (1637s)  ✓ best
      Batch 2857/2857 | Running Loss: 0.1240
  epoch 10/1

 ## CELL 6 — Final Test Evaluation (all three models)



 Loads each best checkpoint, evaluates on the held-out test split.

 Reports overall mAP@0.5, per-class AP, and person precision/recall/F1

 side-by-side for easy comparison.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6  |  FINAL TEST EVALUATION & ERROR ANALYSIS DUMP
# ══════════════════════════════════════════════════════════════════════
import csv
import json
import time
import torch
import gc
from typing import Dict

EVAL_TARGETS = [
    ("Stage 0 (head only)",   STAGE0_CKPT),
    ("Stage 1 (full extras)", STAGE1_CKPT),
    ("Stage 1 (plain)",       STAGE1P_CKPT),
    ("Stage 2 (incremental)", STAGE2_CKPT),
]

test_ds = VOCDataset("test", DetectionTransform("baseline", train=False))
test_dl = DataLoader(test_ds, batch_size=1, shuffle=False,
                     num_workers=2, collate_fn=collate_fn)
print(f"Test split: n={len(test_ds)}")

all_results: Dict[str, dict] = {}

# 1. NEW HELPER FUNCTION FOR ERROR ANALYSIS
def dump_predictions_for_analysis(model, dataloader, device, save_path):
    """Runs inference and saves raw bounding boxes, labels, and scores to JSON."""
    model.eval()
    error_analysis_data = []

    with torch.no_grad():
        for batch_idx, (imgs, tgts) in enumerate(dataloader):
            imgs = [img.to(device) for img in imgs]
            outputs = model(imgs)

            for tgt, out in zip(tgts, outputs):
                # Safely extract image ID (fallback to batch index if missing)
                img_id = tgt.get("image_id", torch.tensor([batch_idx])).cpu().tolist()
                if isinstance(img_id, list):
                    img_id = img_id[0] if len(img_id) > 0 else batch_idx

                error_analysis_data.append({
                    "image_id": img_id,
                    "ground_truth": {
                        "boxes": tgt["boxes"].cpu().tolist(),
                        "labels": tgt["labels"].cpu().tolist()
                    },
                    "predictions": {
                        "boxes": out["boxes"].cpu().tolist(),
                        "labels": out["labels"].cpu().tolist(),
                        "scores": out["scores"].cpu().tolist()
                    }
                })

    with open(save_path, 'w') as f:
        json.dump(error_analysis_data, f)
    return str(save_path)


# 2. MAIN EVALUATION LOOP
for label, ckpt_path in EVAL_TARGETS:
    print(f"\n{'═'*62}")
    print(f"  {label}")
    print(f"  Checkpoint: {ckpt_path}")
    print(f"{'═'*62}")

    if not ckpt_path.is_file():
        print(f"  ⚠ Missing — skipping.")
        continue

    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    saved_epoch = ckpt.get('epoch', 'N/A')
    val_map50   = ckpt.get('map50', ckpt.get('val_map50_final', 0.0))

    print(f"  Saved at epoch {saved_epoch}  val_mAP@0.5={val_map50:.4f}")

    model_eval = get_model().to(DEVICE)
    model_eval.load_state_dict(ckpt["model_state"])
    model_eval.eval()

    # --- Evaluation ---
    t0          = time.time()
    res         = evaluate_map(model_eval, test_dl, DEVICE)
    map50_test  = float(res["map_50"])
    elapsed_map = time.time() - t0

    # --- NEW: Generate Error Analysis JSON ---
    safe_label_name = label.replace(" ", "_").replace("(", "").replace(")", "").lower()
    preds_filename = CKPT_DIR / f"{safe_label_name}_predictions.json"
    print(f"  Generating raw prediction dump for error analysis...")
    dump_predictions_for_analysis(model_eval, test_dl, DEVICE, preds_filename)

    # --- Print Class Breakdown ---
    per_class_ap = res.get("map_per_class", None)
    per_class_dict: Dict[str, float] = {}
    if per_class_ap is not None:
        print(f"\n  Per-class AP@0.5:")
        for cls_idx in range(1, NUM_CLASSES):
            ap = float(per_class_ap[cls_idx - 1])
            per_class_dict[VOC_CLASSES[cls_idx]] = ap

    precision, recall = evaluate_person_pr(model_eval, test_dl, DEVICE, iou_threshold=0.5)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)

    print(f"\n  Overall mAP@0.5 = {map50_test:.4f}  ({elapsed_map:.0f}s)")
    print(f"  Predictions saved to: {preds_filename.name}")

    # --- Save to Summary Dict ---
    all_results[label] = {
        "checkpoint":       str(ckpt_path),
        "saved_epoch":      saved_epoch,
        "val_map50":        val_map50,
        "test_map50":       map50_test,
        "per_class_ap":     per_class_dict,
        "person_precision": precision,
        "person_recall":    recall,
        "person_f1":        f1,
        "error_analysis_file": str(preds_filename) # <--- ADDED REFERENCE HERE
    }

    del model_eval
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# 3. EXPORT FINAL SUMMARIES
print(f"\n{'═'*62}\n  SUMMARY — TEST SET\n{'═'*62}")
for label, r in all_results.items():
    print(f"  {label:<26} Test mAP: {r['test_map50']:.4f}")

summary_json_path = CKPT_DIR / "test_evaluation_summary.json"
summary_json_path.write_text(json.dumps(all_results, indent=2))

summary_csv_path = CKPT_DIR / "test_summary_statistics.csv"
with open(summary_csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Model", "Test mAP@0.5", "Person Precision", "Person Recall", "Person F1", "Predictions File"])
    for label, r in all_results.items():
        writer.writerow([
            label, f"{r['test_map50']:.4f}", f"{r['person_precision']:.4f}",
            f"{r['person_recall']:.4f}", f"{r['person_f1']:.4f}", r['error_analysis_file']
        ])
print(f"\n✅ Deep JSON summary saved → {summary_json_path}")
print(f"✅ CSV statistics saved → {summary_csv_path}")

Test split: n=4952

══════════════════════════════════════════════════════════════
  Stage 0 (head only)
  Checkpoint: checkpoints/stage0_warmup.pth
══════════════════════════════════════════════════════════════
  Saved at epoch 4  val_mAP@0.5=0.8466
  Generating raw prediction dump for error analysis...

  Per-class AP@0.5:

  Overall mAP@0.5 = 0.8471  (842s)
  Predictions saved to: stage_0_head_only_predictions.json

══════════════════════════════════════════════════════════════
  Stage 1 (full extras)
  Checkpoint: checkpoints/stage1_full_finetune_best.pth
══════════════════════════════════════════════════════════════
  Saved at epoch 10  val_mAP@0.5=0.7755
  Generating raw prediction dump for error analysis...

  Per-class AP@0.5:

  Overall mAP@0.5 = 0.7755  (840s)
  Predictions saved to: stage_1_full_extras_predictions.json

══════════════════════════════════════════════════════════════
  Stage 1 (plain)
  Checkpoint: checkpoints/stage1_plain_best.pth
════════════════════════════